In [ ]:
#@title ⚙️ Konfigurasi { display-mode: "form" }
#@markdown Jalankan semua cell berurutan (Runtime → Run all). Secrets opsional (ikon 🔑): `GITHUB_TOKEN` (repo private), `HF_TOKEN`, `GEMINI_API_KEY`, `OPENROUTER_API_KEY`.
REPO_URL = "https://github.com/alpha24-rz/Orlith.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
LLM_MODEL = "qwen2.5:14b"  #@param ["qwen2.5:14b", "qwen2.5:7b", "llama3.1:8b", "gemma3:12b"]
LLM_CONTEXT = 12288  #@param {type:"integer"}
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  #@param ["BAAI/bge-reranker-v2-m3", "BAAI/bge-reranker-base"]
PREFETCH_BAIDU_OCR = True  #@param {type:"boolean"}
PERSIST_TO_DRIVE = True  #@param {type:"boolean"}
FRONTEND_URL = "http://localhost:3000"  #@param {type:"string"}


In [ ]:
#@title 1 · Setup: GPU, kode, dependensi, cache model
import os, sys, time, glob, shutil, sqlite3, subprocess, urllib.request
from concurrent.futures import ThreadPoolExecutor

import torch
assert torch.cuda.is_available(), "GPU tidak terdeteksi → Runtime › Change runtime type › L4 GPU"
_p = torch.cuda.get_device_properties(0)
print(f"🖥️  GPU: {_p.name} · {_p.total_memory / 1024**3:.1f} GB VRAM")
if "L4" not in _p.name:
    print("⚠️  Bukan L4 — kecilkan LLM_MODEL / LLM_CONTEXT bila VRAM < 24 GB.")

ROOT, BACKEND = "/content/Orlith", "/content/Orlith/backend"
DATA, LOGS = "/content/orlith_data", "/content/logs"
DRIVE_DATA = "/content/drive/MyDrive/Orlith/data"
for _d in (DATA, LOGS):
    os.makedirs(_d, exist_ok=True)


def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Gagal: {cmd}")
    return r.stdout


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")


def wait_http(url, timeout, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc is not None and proc.poll() is not None:
            break
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status < 500:
                    return True
        except Exception:
            pass
        time.sleep(2)
    if log and os.path.exists(log):
        print(open(log, errors="ignore").read()[-4000:])
    raise RuntimeError(f"Tidak merespons: {url}")


def backup():
    """Salin data ke Google Drive. File SQLite disalin via backup API (aman saat server berjalan)."""
    if not PERSIST_TO_DRIVE or not os.path.exists(f"{DATA}/documind.db"):
        return
    os.makedirs(DRIVE_DATA, exist_ok=True)
    sh(f"rsync -a --delete --exclude='*.db' --exclude='*.sqlite3' --exclude='*-wal' "
       f"--exclude='*-shm' --exclude='*-journal' {DATA}/ {DRIVE_DATA}/", check=False)
    for src in glob.glob(f"{DATA}/**/*.db", recursive=True) + glob.glob(f"{DATA}/**/*.sqlite3", recursive=True):
        dst = src.replace(DATA, DRIVE_DATA, 1)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        s, d = sqlite3.connect(src), sqlite3.connect(dst + ".tmp")
        s.backup(d); d.close(); s.close()
        os.replace(dst + ".tmp", dst)


# Kode backend (git clone / update)
_gh = secret("GITHUB_TOKEN")
_url = REPO_URL.replace("https://", f"https://{_gh}@") if _gh else REPO_URL
if os.path.isdir(f"{ROOT}/.git"):
    sh(f"git -C {ROOT} fetch --depth 1 origin {BRANCH} && git -C {ROOT} reset --hard FETCH_HEAD")
else:
    sh(f"git clone --depth 1 -b {BRANCH} {_url} {ROOT}")
print("📦 Kode:", sh(f"git -C {ROOT} log -1 --format='%h · %s'").strip())

# Data persisten (Google Drive → lokal; SQLite di Drive langsung lambat & rawan lock)
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(DRIVE_DATA) and not os.path.exists(f"{DATA}/documind.db"):
        sh(f"rsync -a {DRIVE_DATA}/ {DATA}/")
        print("💾 Data dipulihkan dari Drive")
for _d in ("chroma", "uploads"):
    os.makedirs(f"{DATA}/{_d}", exist_ok=True)

# Dependensi (pakai torch CUDA bawaan Colab — tidak diinstal ulang)
print("⏳ Menginstal dependensi...")
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils rsync > /dev/null")
sh("pip -q install uv")
_pkgs = f"-r {BACKEND}/requirements.txt 'transformers>=4.48,<5' pymupdf einops addict easydict"
if subprocess.run(f"uv pip install --system -q {_pkgs}", shell=True).returncode != 0:
    sh(f"pip install -q {_pkgs}")
if not shutil.which("cloudflared"):
    sh("wget -q -O /usr/local/bin/cloudflared "
       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
       "&& chmod +x /usr/local/bin/cloudflared")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh")

# Unduh bobot model HF lebih dulu (tanpa memuat ke VRAM) agar request pertama tidak timeout.
# Mengamankan tqdm._lock agar tidak bentrok dengan notebook environment Colab.
import threading
try:
    import tqdm.std, tqdm.notebook
    tqdm.std.tqdm.get_lock()
    tqdm.notebook.tqdm_notebook.get_lock()
    if not hasattr(tqdm.std.tqdm, "_lock"):
        tqdm.std.tqdm._lock = threading.RLock()
    if not hasattr(tqdm.notebook.tqdm_notebook, "_lock"):
        tqdm.notebook.tqdm_notebook._lock = threading.RLock()
except Exception:
    pass

from huggingface_hub import snapshot_download
_hf = secret("HF_TOKEN") or None
_models = ["BAAI/bge-m3", RERANKER_MODEL] + (["baidu/Unlimited-OCR"] if PREFETCH_BAIDU_OCR else [])
_skip = ["*.onnx", "onnx/*", "*.msgpack", "*.h5", "*.ot", "imgs/*"]
for _m in _models:
    print(f"📥 Sinkronisasi cache model: {_m} ...")
    snapshot_download(_m, token=_hf, ignore_patterns=_skip)
print("✅ Setup selesai:", ", ".join(_models))


In [ ]:
#@title 2 · Ollama (LLM di GPU)
subprocess.run("pkill -f 'ollama serve'", shell=True)
time.sleep(1)
OLLAMA_ENV = {
    **os.environ,
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_FLASH_ATTENTION": "1",        # attention lebih cepat & hemat VRAM
    "OLLAMA_KV_CACHE_TYPE": "q8_0",       # KV cache ~50% lebih kecil
    "OLLAMA_CONTEXT_LENGTH": str(LLM_CONTEXT),  # default Ollama 4K memotong konteks RAG
    "OLLAMA_NUM_PARALLEL": "2",           # HyDE/rewrite + jawaban bisa paralel
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_KEEP_ALIVE": "-1",            # model tetap di VRAM
}
ollama_proc = subprocess.Popen(
    ["ollama", "serve"], env=OLLAMA_ENV, start_new_session=True,
    stdout=open(f"{LOGS}/ollama.log", "w"), stderr=subprocess.STDOUT,
)
wait_http("http://127.0.0.1:11434/api/version", 60, ollama_proc, f"{LOGS}/ollama.log")

print(f"📥 Pull {LLM_MODEL} ...")
sh(f"ollama pull {LLM_MODEL}")

# Muat model ke VRAM sekarang (bukan saat request pertama)
import json as _json
_req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=_json.dumps({"model": LLM_MODEL, "prompt": "ok", "keep_alive": -1,
                      "options": {"num_predict": 1}}).encode(),
    headers={"Content-Type": "application/json"},
)
urllib.request.urlopen(_req, timeout=600).read()
print(sh("ollama ps"))


In [ ]:
#@title 3 · Jalankan backend + Cloudflare tunnel
import re, secrets
from IPython.display import display, HTML

_key_file = f"{DATA}/.secret_key"  # disimpan agar login tetap valid antar sesi
if not os.path.exists(_key_file):
    open(_key_file, "w").write(secrets.token_hex(32))

BACKEND_ENV = {
    "ENVIRONMENT": "development",
    "SECRET_KEY": open(_key_file).read().strip(),
    "DATABASE_URL": f"sqlite+aiosqlite:///{DATA}/documind.db",
    "CHROMA_PERSIST_DIR": f"{DATA}/chroma",
    "STORAGE_BACKEND": "local",
    "STORAGE_LOCAL_PATH": f"{DATA}/uploads",
    "LLM_PROVIDER": "ollama",
    "LLM_MODEL": LLM_MODEL,
    "OLLAMA_BASE_URL": "http://127.0.0.1:11434",
    "OLLAMA_TIMEOUT": "300",
    "EMBEDDING_PROVIDER": "huggingface",
    "EMBEDDING_MODEL": "BAAI/bge-m3",
    "ENABLE_RERANKER": "true",
    "RERANKER_MODEL": RERANKER_MODEL,
    "ENABLE_SEMANTIC_CHUNKING": "true",
    "ENABLE_HYBRID_SEARCH": "true",
    "ENABLE_PARENT_CHILD_CHUNKING": "true",
    "ENABLE_HYDE": "true",
    "ENABLE_CHUNK_STITCHING": "true",
    "RAG_CANDIDATE_POOL_SIZE": "40",
    "INGESTION_CONCURRENCY": "2",
    "BAIDU_OCR_MODEL_PATH": "baidu/Unlimited-OCR",
    "CORS_ORIGINS": "*",
    "LOG_LEVEL": "INFO",
    "GEMINI_API_KEY": secret("GEMINI_API_KEY"),
    "OPENROUTER_API_KEY": secret("OPENROUTER_API_KEY"),
    "HF_TOKEN": secret("HF_TOKEN"),
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
}
BACKEND_ENV = {k: v for k, v in BACKEND_ENV.items() if v}
with open(f"{ROOT}/.env", "w") as f:  # dibaca oleh backend/core/config.py
    f.write("\n".join(f"{k}={v}" for k, v in BACKEND_ENV.items()) + "\n")

subprocess.run("pkill -f 'uvicorn main:app'; pkill -f cloudflared", shell=True)
time.sleep(1)
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000",
     "--timeout-keep-alive", "75"],
    cwd=BACKEND, env={**os.environ, **BACKEND_ENV}, start_new_session=True,
    stdout=open(f"{LOGS}/backend.log", "w"), stderr=subprocess.STDOUT,
)
wait_http("http://127.0.0.1:8000/health", 180, backend_proc, f"{LOGS}/backend.log")
print("✅ Backend aktif di :8000")

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    start_new_session=True, stdout=open(f"{LOGS}/tunnel.log", "w"), stderr=subprocess.STDOUT,
)
PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOGS}/tunnel.log", errors="ignore").read())
    if _m:
        PUBLIC_URL = _m.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError(f"URL tunnel tidak ditemukan — cek {LOGS}/tunnel.log")
try:
    wait_http(f"{PUBLIC_URL}/health", 45)
except RuntimeError:
    print("⚠️  Tunnel belum bisa diakses (DNS masih propagasi) — coba lagi sebentar.")

_link = f"{FRONTEND_URL.rstrip('/')}/?backend={PUBLIC_URL}"
display(HTML(f"""
<div style="font-family:system-ui;padding:14px 18px;border-radius:12px;background:#0f172a;color:#e2e8f0;line-height:1.7">
  <div style="font-size:15px;font-weight:600">🚀 Orlith backend online</div>
  <div>Backend URL: <code style="color:#a5b4fc">{PUBLIC_URL}</code></div>
  <div>Hubungkan frontend: <a href="{_link}" target="_blank" style="color:#34d399">{_link}</a></div>
  <div style="color:#94a3b8;font-size:12px">Atau tempel Backend URL di halaman <code>/status</code> frontend ·
    <a href="{PUBLIC_URL}/docs" target="_blank" style="color:#93c5fd">API docs</a></div>
</div>"""))


In [ ]:
#@title 4 · Monitor & auto-backup (biarkan berjalan; Stop = server tetap hidup)
BACKUP_EVERY_MIN = 10  #@param {type:"integer"}
import json as _json
_last = time.time()
try:
    while True:
        if backend_proc.poll() is not None:
            print("\n❌ Backend berhenti. Log terakhir:\n")
            print(open(f"{LOGS}/backend.log", errors="ignore").read()[-4000:])
            break
        try:
            _h = _json.load(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=10))
            _g = _h.get("gpu") or {}
            _gpu = f"{_g.get('memory_used_gb', 0):.1f}/{_g.get('memory_total_gb', 0):.1f} GB" if _g else "-"
            _st = _h.get("status", "?")
        except Exception as e:
            _st, _gpu = f"error ({e.__class__.__name__})", "-"
        print(f"\r[{time.strftime('%H:%M:%S')}] backend: {_st} · VRAM {_gpu} · {PUBLIC_URL}   ", end="")
        if time.time() - _last > BACKUP_EVERY_MIN * 60:
            backup()
            _last = time.time()
        time.sleep(30)
except KeyboardInterrupt:
    backup()
    print("\n⏸️  Monitor dihentikan, data dibackup. Server tetap berjalan.")


In [ ]:
#@title 🧾 Lihat log
LOG = "backend"  #@param ["backend", "ollama", "tunnel"]
LINES = 80  #@param {type:"integer"}
print(sh(f"tail -n {LINES} {LOGS}/{LOG}.log", check=False))


In [ ]:
#@title 🛑 Stop server & backup ke Drive
backup()
subprocess.run("pkill -f 'uvicorn main:app'; pkill -f cloudflared; pkill -f 'ollama serve'", shell=True)
print("🛑 Semua proses dihentikan" + (" · data tersimpan di Drive" if PERSIST_TO_DRIVE else ""))
